# Citi Bike: tuned ridge regression

**Contribution owner: Tomislav Novosel.** Implementation, experiments and initial explanations were prepared with AI assistance on 9 October 2026. Human review and substantive changes should be recorded when they happen; see [the assistance record](../docs/AI_USE.md).

These are development experiments, not final-test results. The training/tuning dates end in June; July-September is development validation. October-December has not been scored.

The searches were executed with the helper scripts and their actual reports are saved. By default this notebook reviews those reports quickly. Set `CITIBIKE_RETRAIN=1` or use the runner's `--train` option to execute the search again. The XGBoost search needs a CUDA GPU; viewing results and running the saved app do not.


Ridge regression combines numeric features with a penalty that limits unstable coefficients. Scaling is fitted separately inside each training fold.

In [7]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd()
while not (ROOT / 'src/citibike_forecast.py').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open the notebook inside the repository.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
import numpy as np
from citibike_forecast import REPORTS, MODELS, development_data, chronological_cv
RUN_TRAINING = os.environ.get('CITIBIKE_RETRAIN') == '1'
def report(name):
    return json.loads((REPORTS / (name + '_run.json')).read_text(encoding='utf8'))
print('Mode:', 'execute training' if RUN_TRAINING else 'review actual saved experiments')
from citibike_forecast import run_tuning
if RUN_TRAINING:
    run_tuning('ridge')
run = report('ridge_tuned')
display(pd.Series(run['search']))
grid = pd.read_csv(REPORTS / 'ridge_grid.csv').sort_values('rank_test_score')
display(grid.filter(regex='param_|mean_test_score|rank_test_score').head(10))


Mode: review actual saved experiments


method                                                             GridSearchCV
configurations                                                               28
cv_fits                                                                      84
best_parameters               {'feature_set': 'compact', 'model__ridge__alph...
best_cv_mae                                                        11604.782036
elapsed_seconds                                                        3.095465
actual_fitted_device                                                       None
validation_used_for_search                                                False
dtype: object

,param_feature_set,param_model__ridge__alpha,param_seasonal_residual,mean_test_score,rank_test_score
6,compact,10.0,False,-11604.782036,1
7,compact,10.0,True,-12073.216147,2
22,full,100.0,False,-12820.187111,3
20,full,10.0,False,-13106.865308,4
8,compact,100.0,False,-13157.192664,5
21,full,10.0,True,-13233.830819,6
5,compact,1.0,True,-13442.318931,7
4,compact,1.0,False,-13499.939761,8
18,full,1.0,False,-13878.729581,9
19,full,1.0,True,-13884.446341,10


We searched seven penalties, two feature sets and two target formulations: absolute rides or change from last week. That makes 28 configurations, each checked on three chronological folds.

In [8]:
display(pd.Series(run['validation']))
print('The selected configuration matches the reference ridge model. The search did not create a new improvement.')
print('Its average training-CV error is lower than XGBoost, but the later validation result is worse. This shows how the season and evaluation dates matter.')


The selected configuration matches the reference ridge model. The search did not create a new improvement.
Its average training-CV error is lower than XGBoost, but the later validation result is worse. This shows how the season and evaluation dates matter.


mae     17850.334487
rmse    23119.476403
wape        0.148012
bias    -6392.475609
days       92.000000
dtype: float64